# [실습4] 머신러닝을 활용한 양극소재 기반 배터리 특성 예측(실습)
---

## 실습 목표

- 머신러닝 분류 모델들의 원리를 이해하고 코드를 작성합니다.
- 주성분 분석을 통해 저차원 공간에서 피처가 분리되는 것을 확인합니다.
- 분류 모델들의 성능을 비교해봅니다. 각각 토이 데이터셋과 실습 데이터셋에서 검증합니다.

---

## 실습 목차

1. **토이 데이터셋 전처리** 

2. **주성분 분석(PCA)** 

3. **로지스틱 회귀 모델**

4. **결정 트리** 

5. **K-최근접 이웃(KNN)** 

6. **랜덤 포레스트(Random Forest)** 

7. **XGBoost** 

8. **실습 데이터 불러오기 및 분류 모델 적용** 
---

## 실습 개요

이번 실습에서는 머신러닝 분류 모델들을 구현하고 실습 데이터에 어떻게 적용될 수 있는지 확인합니다.

---

## 1. 토이 데이터셋 전처리
---
우선 타이타닉 토이 데이터셋을 이용해 머신러닝 알고리즘들을 적용해보겠습니다.

In [ ]:
## 라이브러리 불러오기
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt 

from warnings import simplefilter
simplefilter(action='ignore', category=FutureWarning)

---
### 1.1 토이 데이터셋 불러오기
타이타닉 데이터셋을 불러옵니다.

In [ ]:
df = pd.read_csv('./data/titanic_processed.csv') # df: dataframe의 줄임말
df.drop(columns = df.columns[0], axis = 1, inplace= True) # 데이터 순서를 표현하는 첫 열을 제거합니다.
display(df)

해당 승객의 생존 여부를 정답 데이터로 지정하겠습니다. 나머지 데이터를 입력 데이터로 지정합니다.

In [ ]:
from sklearn.model_selection import train_test_split

X = df[['pclass', 'age', 'sibsp', 'parch', 'fare', 'gender', 'grade']].astype('float')

y = df['survived']

display(X)
display(y)

---
### 1.2 정규화 & 표준화
토이 데이터셋에 정규화와 표준화를 진행합니다. 

In [ ]:
from sklearn.preprocessing import StandardScaler
std_scaler = StandardScaler()
std_scaler.fit(X)
scaled_X = std_scaler.transform(X)
X = pd.DataFrame(scaled_X, columns=X.columns, index=list(X.index.values))
display(X)

---
### 1.3 학습 & 테스트 데이터 구분

In [ ]:

random_seed = 42
X_train, X_test, Y_train, Y_test = train_test_split(X, y, test_size = 0.3, random_state = random_seed)


---

## 2. 주성분 분석(PCA)
토이 데이터셋에 PCA 알고리즘을 적용해봅니다. 

PCA는 Principal Component Analysis로, 피처의 차원을 축소하여 원하는 개수의 주성분으로 표현하는 기법입니다. 

이는 고차원의 데이터를 저차원의 데이터로 변환하는 차원 축소 방법 중 하나입니다.

---
### 2.1 2차원 PCA 
데이터를 2차원으로 축소합니다. 즉, 주성분이 2개로 이뤄져있습니다.

In [ ]:
from sklearn.decomposition import PCA

def get_pca_data(ss_data, n_components = 2):
    pca = PCA(n_components = n_components)
    pca.fit(ss_data)
    
    return pca.transform(ss_data), pca

PCA를 적용하여 차원을 축소합니다.

In [ ]:
pca_data, pca = get_pca_data(X_train, n_components=2)

### 2.2 2차원 PCA 결과 시각화
축소된 차원을 2차원 그래프로 시각화합니다.

In [ ]:
import seaborn as sns

pca_columns = ['pca_1', 'pca_2']
pca_pd = pd.DataFrame(pca_data, columns=pca_columns)
pca_pd['survived'] = Y_train

sns.pairplot(pca_pd, hue='survived', height=5,
             x_vars=['pca_1'], y_vars=['pca_2'])

plt.show()

2차원 그래프로는 생존 여부에 따라 피처가 잘 구분되지 않는 것을 확인할 수 있습니다.

---
### 2.3 3차원 PCA
데이터를 3차원으로 축소합니다. 즉, 주성분이 3개로 이뤄져있습니다.

### [TODO] 위 설명을 참고하여 적절하게 _____를 수정행보세요.

In [ ]:
pca_data, pca = get_pca_data(X_train, n_components=___)

시각화를 위해 pandas 데이터 형태로 변환하는 함수를 선언합니다.

In [ ]:
def get_pd_from_pca(pca_data, col_num):
    cols = ['pca_'+str(n) for n in range(col_num)]
    return pd.DataFrame(pca_data, columns = cols)

In [ ]:
pca_pd = get_pd_from_pca(pca_data, 3)

pca_pd['survived'] = Y_train.values
pca_pd.head()

---
### 2.4 3차원 PCA 결과 시각화
축소된 차원을 3차원 그래프로 시각화합니다.

In [ ]:
from mpl_toolkits.mplot3d import Axes3D

markers = ['^', 'o']

fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')

for i, marker in enumerate(markers):
    x_axis_data = pca_pd[pca_pd['survived'] == i]['pca_0']
    y_axis_data = pca_pd[pca_pd['survived'] == i]['pca_1']
    z_axis_data = pca_pd[pca_pd['survived'] == i]['pca_2']

    ax.scatter(x_axis_data, y_axis_data, z_axis_data,
               s=20, alpha=0.5, marker=marker, label=str(i))
    
ax.view_init(30, 80)
plt.legend()
plt.show()

2차원 PCA 결과보다 데이터가 축소 차원 공간에서 더 잘 구분되는 것을 확인할 수 있습니다.

PCA 결과를 통해 간단한 머신러닝 알고리즘의 예측 성능을 간접적으로 유추해볼 수 있습니다.

2차원 혹은 3차원 PCA 결과로 축소 차원 공간에서 데이터가 완벽하게 분류된다면, 매우 간단한 알고리즘으로도 높은 정확도로 분류를 할 수 있습니다.

이와 반대 상황이라면, 보다 복잡하고 비선형성이 강한 알고리즘을 선택해야 합니다.

## 3. 로지스틱 회귀 모델
이전 실습에서 적용했었던 로지스틱 회귀 모델을 토이 데이터셋에 동일하게 적용해보겠습니다.

추후 다른 알고리즘들과 성능을 비교합니다.

---
### 3.1 로지스틱 회귀 모델 학습

### [TODO] 라이브러리 이름을 확인하여 적절하게 _____를 수정행보세요.

In [ ]:
from sklearn.linear_model import LogisticRegression #logistic regression
model = ______Regression()
model.fit(X_train, Y_train)

---
### 3.2 모델 성능 확인
생존 여부의 예측 성능을 확인합니다.

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

def evaluation(prediction, Y_test, model_name="모델 이름"):
    print(model_name, "의 정답률은 %.2f%% 입니다." % (accuracy_score(prediction, Y_test) * 100))
    print(model_name, "의 정밀도는 %.2f 입니다." % precision_score(Y_test, prediction))
    print(model_name, "의 재현율은 %.2f 입니다." % recall_score(Y_test, prediction))
    print(model_name, "의 F1 점수는 %.2f 입니다." % f1_score(Y_test, prediction))
    print(model_name, "의 ROC-AUC 점수는 %.2f 입니다." % roc_auc_score(Y_test, prediction))

In [ ]:
LR_prediction = model.predict(X_test)
evaluation(LR_prediction, Y_test, model_name="로지스틱 회귀 모델")

타이타닉 영화의 두 주인공이 생존할 수 있었을지 예측해봅니다.

Decaprio와 Winslet의 영화 설정 상 데이터를 구성해주고, 이를 함수로 정의합니다.

모든 알고리즘에서 생존 여부를 출력해보겠습니다.

In [ ]:
def check_survive(model):
    # 'pclass', 'age', 'sibsp', 'parch', 'fare', 'gender', 'grade'
    decaprio = np.array([[3, 18, 0, 0, 5, 1, 1]])
    decaprio = pd.DataFrame(decaprio, columns=X.columns)
    decaprio = std_scaler.transform(decaprio)
    decaprio = pd.DataFrame(decaprio, columns=X.columns)
    print('Decaprio : ', model.predict(decaprio)[0])

    winslet = np.array([[1, 16, 1, 1, 100, 0, 3]])
    winslet = pd.DataFrame(winslet, columns=X.columns)
    winslet = std_scaler.transform(winslet)
    winslet = pd.DataFrame(winslet, columns=X.columns)
    print('Winslet : ', model.predict(winslet)[0])


In [ ]:
check_survive(model)

Decaprio는 생존하지 못하고, Winslet은 생존을 하는 것으로 예측되었습니다.

---
## 4. 결정 트리
---
### 4.1 결정 트리 모델 학습

In [ ]:
from sklearn.tree import DecisionTreeClassifier #Decision Tree

model = DecisionTreeClassifier()
model.fit(X_train, Y_train)


---
### 4.2 모델 성능 확인
생존 여부의 예측 성능을 확인합니다.

In [ ]:
DT_prediction = model.predict(X_test)
evaluation(DT_prediction, Y_test, model_name="결정 트리 모델")

In [ ]:
check_survive(model)

---
## 5. K-최근접 이웃(KNN)
---
### 5.1 KNN 모델 학습

In [ ]:
from sklearn.neighbors import KNeighborsClassifier #KNN

model = KNeighborsClassifier(n_neighbors=9)
model.fit(X_train, Y_train)

---
### 5.2 모델 성능 확인
생존 여부의 예측 성능을 확인합니다.

### [TODO] 결정트리의 분석 사례를 참고하여 적절하게 _____를 수정행보세요.

In [ ]:
KNN_prediction = model.predict(______)
evaluation(KNN_prediction, ______, model_name="KNN")

In [ ]:
check_survive(model)

---
### 5.3 이웃 파라미터에 따른 성능 분석
KNN은 이웃 파라미터의 크기에 따라 성능이 크게 좌우되는 알고리즘입니다.

이웃 파라미터를 조정하면서 성능을 확인해보겠습니다.

In [ ]:
k_list = np.arange(1,15)
acc_list = []
for i in k_list:
    model = KNeighborsClassifier(n_neighbors=i)
    model.fit(X_train, Y_train)
    prediction = model.predict(X_test)
    acc_list.append(accuracy_score(prediction, Y_test))


In [ ]:
acc_list = np.array(acc_list)
x = np.arange(0,15)
plt.plot(k_list, acc_list)
plt.xticks(x)
plt.show()
print('가장 높은 정답률을 보이는 이웃 파라미터는 ', np.argmax(acc_list)+1, '입니다.')

---
## 6. 랜덤 포레스트(Random Forest)
---
### 6.1 랜덤 포레스트 모델 학습

In [ ]:
from sklearn.ensemble import RandomForestClassifier #Random Forest

model = RandomForestClassifier()
model.fit(X_train, Y_train)

---
### 6.2 모델 성능 확인
생존 여부의 예측 성능을 확인합니다.

In [ ]:
RF_prediction = model.predict(X_test)
evaluation(RF_prediction, Y_test, model_name="랜덤 포레스트 모델")

In [ ]:
check_survive(model)

---
## 7. XGBoost
---
### 7.1 XGBoost 모델 학습

In [ ]:
import xgboost as XGBRegressor
model = XGBRegressor.XGBClassifier(n_estimators=900,learning_rate=0.1)
model.fit(X_train, Y_train)

---
### 7.2 모델 성능 확인
생존 여부의 예측 성능을 확인합니다.

In [ ]:
XGB_prediction = model.predict(X_test)
evaluation(XGB_prediction, Y_test, model_name="XGBoost 모델")

In [ ]:
check_survive(model)

모든 모델의 성능을 바 그래프로 시각화하여 비교해보겠습니다. 

### [TODO] 시각화할 항목을 확인하여 적절하게 _____를 수정행보세요.

In [ ]:
model_list = ['Logistic', 'Decision Tree', 'KNN', 'Random Forest', 'XGBoost']
result_list = [accuracy_score(LR_prediction, Y_test)*100,
               accuracy_score(DT_prediction, Y_test)*100,
               accuracy_score(KNN_prediction, Y_test)*100,
               accuracy_score(RF_prediction, Y_test)*100,
               accuracy_score(_____prediction, Y_test)*100
              ] 

bar = plt.bar(model_list, result_list)
for rect in bar:
    height = rect.get_height()
    plt.text(rect.get_x() + rect.get_width()/2.0, height, '%.1f' % height, ha='center', va='bottom', size = 12)
plt.ylim(0, 110)
plt.show()      

알고리즘 별 성능의 차이가 크지 않은 것으로 확인됩니다.

---
## 8. 실습 데이터 불러오기 및 분류 모델 적용
머신러닝 분류 모델들을 실습 데이터에 적용해보겠습니다.

---
### 8.1 실습 데이터 불러오기
이전 실습에서 모든 전처리가 끝난 실습데이터를 불러오겠습니다.

In [ ]:
df = pd.read_csv('./data/processed.csv') # df: dataframe의 줄임말
df.drop(columns = df.columns[0], axis = 1, inplace= True) # 데이터 순서를 표현하는 첫 열을 제거합니다.
display(df)

---
### 8.2 학습 & 테스트 데이터 구분
7:3 의 비율로 학습 데이터와 테스트 데이터를 구분합니다.

이때 Crystal System 데이터를 정답 데이터로 분리합니다.

In [ ]:
target_col = 'Crystal System'

y = df[target_col]
y = y.astype('category').cat.codes
X = df.drop(target_col, axis=1)
X_train, X_test, Y_train, Y_test = train_test_split(X, y, test_size=0.3, random_state=random_seed)

print(X_train.shape)
print(X_test.shape)
print(Y_train.shape)
print(Y_test.shape)

---
### 8.3 2차원 PCA 적용

In [ ]:
pca_data, pca = get_pca_data(X_train, n_components=2)

In [ ]:
import seaborn as sns

pca_columns = ['pca_1', 'pca_2']
pca_pd = pd.DataFrame(pca_data, columns=pca_columns)
pca_pd['Crystal System'] = Y_train

sns.pairplot(pca_pd, hue='Crystal System', height=5,
             x_vars=['pca_1'], y_vars=['pca_2'])

plt.show()

---
### 8.4 3차원 PCA 적용

### [TODO] 3차원 PCA 를 적용하기 위해 _____를 수정해보세요.

In [ ]:
pca_data, pca = get_pca_data(X_train, n_components=___)

In [ ]:
pca_pd = get_pd_from_pca(pca_data, 3)

pca_pd['Crystal System'] = Y_train.values
pca_pd.head()

In [ ]:
from mpl_toolkits.mplot3d import Axes3D

markers = ['^', 'o', '*']

fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')

for i, marker in enumerate(markers):
    x_axis_data = pca_pd[pca_pd['Crystal System'] == i]['pca_0']
    y_axis_data = pca_pd[pca_pd['Crystal System'] == i]['pca_1']
    z_axis_data = pca_pd[pca_pd['Crystal System'] == i]['pca_2']

    ax.scatter(x_axis_data, y_axis_data, z_axis_data,
               s=50, alpha=0.5, marker=marker, label=str(i))
    
ax.view_init(30, 60)
plt.legend()
plt.show()

---
### 8.5 로지스틱 회귀 모델 적용

In [ ]:
# 다중 클래스 분류를 위한 로지스틱 회귀 모델
model = LogisticRegression(multi_class='multinomial', solver='lbfgs')
model.fit(X_train, Y_train)


실습 데이터는 정답 데이터의 클래스가 3개입니다. 즉, 다중 분류(Multi-Classs) 문제입니다.

이때, 이진 분류 문제의 평가 지표들을 그대로 적용할 수는 없습니다.

따라서, 성능 지표 함수에 다중 분류 성능을 어떻게 평가할지 지정해줘야 합니다.

본 실습에서는 가장 많이 사용하는 weighted 방법을 사용하겠습니다.

In [ ]:
def evaluation(prediction, Y_test, model_name="모델 이름"):
    print(model_name, "의 정답률은 %.2f%% 입니다." % (accuracy_score(prediction, Y_test) * 100))
    print(model_name, "의 정밀도는 %.2f 입니다." % precision_score(Y_test, prediction, average='weighted'))
    print(model_name, "의 재현율은 %.2f 입니다." % recall_score(Y_test, prediction, average='weighted'))
    print(model_name, "의 F1 점수는 %.2f 입니다." % f1_score(Y_test, prediction, average='weighted'))

AUC-ROC의 경우, 다중 분류 문제일 경우 각 클래스 별 예측 확률이 필요합니다.

별도의 함수를 만들어 처리합니다.

In [ ]:
def multi_class_auc(prediction_prob, Y_test, model_name="모델 이름"):
    print(model_name, "의 ROC-AUC 점수는 %.2f 입니다." % roc_auc_score(Y_test, prediction_prob, multi_class='ovr'))  

In [ ]:
LR_prediction = model.predict(X_test)
LR_prediction_prob = model.predict_proba(X_test)
evaluation(LR_prediction, Y_test, model_name="로지스틱 회귀 모델")
multi_class_auc(LR_prediction_prob, Y_test, model_name="로지스틱 회귀 모델")

---
### 8.6 결정 트리 모델 적용

In [ ]:

model = DecisionTreeClassifier()
model.fit(X_train, Y_train)

In [ ]:
DT_prediction = model.predict(X_test)
DT_prediction_prob = model.predict_proba(X_test)
evaluation(DT_prediction, Y_test, model_name="결정 트리 모델")
multi_class_auc(DT_prediction_prob, Y_test, model_name="결정 트리 모델")

---
### 8.7 KNN 모델 적용

In [ ]:
model = KNeighborsClassifier(n_neighbors=15)
model.fit(X_train, Y_train)

In [ ]:
KNN_prediction = model.predict(X_test)
KNN_prediction_prob = model.predict_proba(X_test)
evaluation(KNN_prediction, Y_test, model_name="KNN")
multi_class_auc(KNN_prediction_prob, Y_test, model_name="KNN")

이웃 파라미터를 조정하면서 성능을 확인해보겠습니다.

In [ ]:
k_list = np.arange(1,20)
acc_list = []
for i in k_list:
    model = KNeighborsClassifier(n_neighbors=i)
    model.fit(X_train, Y_train)
    prediction = model.predict(X_test)
    acc_list.append(accuracy_score(prediction, Y_test))

acc_list = np.array(acc_list)
x = np.arange(0,20)
plt.plot(k_list, acc_list)
plt.xticks(x)
plt.show()
print('가장 높은 정답률을 보이는 이웃 파라미터는 ', np.argmax(acc_list)+1, '입니다.')

---
### 8.8 랜덤 포레스트 모델 적용

In [ ]:
model = RandomForestClassifier()
model.fit(X_train, Y_train)

In [ ]:
RF_prediction = model.predict(X_test)
RF_prediction_prob = model.predict_proba(X_test)
evaluation(RF_prediction, Y_test, model_name="랜덤 포레스트 모델")
multi_class_auc(RF_prediction_prob, Y_test, model_name="랜덤 포레스트 모델")

---
### 8.9 XGBoost 모델 적용

In [ ]:
model = XGBRegressor.XGBClassifier(n_estimators=900,learning_rate=0.1)
model.fit(X_train, Y_train)


In [ ]:
XGB_prediction = model.predict(X_test)
XGB_prediction_prob = model.predict_proba(X_test)
evaluation(XGB_prediction, Y_test, model_name="XGBoost 모델")
multi_class_auc(XGB_prediction_prob, Y_test, model_name="XGBoost 모델")

### [TODO] 시각화할 항목을 확인하고  _____를 수정해보세요.


In [ ]:
model_list = ['Logistic', 'Decision Tree', 'KNN', 'Random Forest', 'XGBoost']
result_list = [accuracy_score(LR_prediction, Y_test)*100,
               accuracy_score(DT_prediction, Y_test)*100,
               accuracy_score(____prediction, Y_test)*100,
               accuracy_score(RF_prediction, Y_test)*100,
               accuracy_score(XGB_prediction, Y_test)*100
              ] 

bar = plt.bar(model_list, result_list)
for rect in bar:
    height = rect.get_height()
    plt.text(rect.get_x() + rect.get_width()/2.0, height, '%.1f' % height, ha='center', va='bottom', size = 12)
plt.ylim(0, 110)
plt.show()      

In [ ]:
model_list = ['Logistic', 'Decision Tree', 'KNN', 'Random Forest', 'XGBoost']
result_list = [roc_auc_score(Y_test, LR_prediction_prob, multi_class='ovr'),
               roc_auc_score(Y_test, DT_prediction_prob, multi_class='ovr'),
               roc_auc_score(Y_test, KNN_prediction_prob, multi_class='ovr'),
               roc_auc_score(Y_test, RF_prediction_prob, multi_class='ovr'),
               roc_auc_score(Y_test, XGB_prediction_prob, multi_class='ovr')
              ] 

bar = plt.bar(model_list, result_list)
for rect in bar:
    height = rect.get_height()
    plt.text(rect.get_x() + rect.get_width()/2.0, height, '%.2f' % height, ha='center', va='bottom', size = 12)
plt.ylim(0, 1.1)
plt.show()     

알고리즘 별 성능을 확인했을 때, 정답률와 AUC-ROC 두 지표에서 모두 좋은 성능을 보이고 있습니다. 

특히, 결정 트리, 랜덤 포레스트, XGBoost 세 알고리즘이 실습 데이터에 대해서 뛰어난 예측 성능을 보이는 것을 확인할 수 있습니다.